# 🧪 Taller práctico: Análisis y Modelado de Series de Tiempo con ARIMA y SARIMA


En este taller aplicarás los conceptos fundamentales del análisis de series de tiempo, desde la exploración hasta el ajuste de modelos ARIMA y SARIMA.

Trabajarás con una serie real: nacimientos diarios de mujeres en California (1959).

Que se encuentra en este link: https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-total-female-births.csv

## 1️⃣ Carga y preparación de los datos

**1.1** Carga el archivo CSV `daily-total-female-births.csv` en un DataFrame.

In [ ]:
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-total-female-births.csv')

In [ ]:
df.tail()

**1.2** Convierte la columna de fechas al tipo `datetime` y establécela como índice.

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace = True)

**1.3** Visualiza las primeras filas del DataFrame para asegurarte de que todo esté en orden.

In [ ]:
df.head()

## 2️⃣ Exploración inicial de la serie de tiempo

**2.1** Grafica la serie completa. ¿Notas tendencias, picos o estacionalidad?

In [ ]:
df['1959-01': '1959-03'].plot(title = 'Nacimientos en California en 1959', ylabel = 'Nacimientos')

**2.2** Resamplea la serie por mes y grafica el promedio mensual. ¿Qué meses presentan los valores más altos en promedio?

In [ ]:
import matplotlib.pyplot as plt

df.resample('M').mean().plot()
plt.grid()

**2.3** Agrupa por trimestre (Q1, Q2, Q3, Q4) y encuentra qué trimestres tienen más nacimientos en promedio.

In [ ]:
df.resample('Q').mean().plot()
plt.grid()

**2.4** Calcula y grafica el promedio móvil (7 días) y la desviación estándar móvil.

In [ ]:
df['Moving Average'] = df['Births'].rolling(7).mean()
df['Moving STD'] = df['Births'].rolling(7).std()
df.plot()

## 3️⃣ Descomposición estacional

**3.1** Aplica una descomposición aditiva con un período de 30 días.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

In [ ]:
serie = df['Births']

In [ ]:
descomposition = seasonal_decompose(df['Births'], model = 'additive', period = 30)
descomposition_plot = descomposition.plot()

In [ ]:
import numpy as np

In [ ]:
descomposition.seasonal['1959-01': '1959-03'].plot()

In [ ]:
np.abs(descomposition.resid.resample('M').mean()).plot()
descomposition.resid.resample('M').mean().plot()

**3.2** Grafica los componentes: tendencia, estacionalidad y residuos.

In [ ]:
descomposition.seasonal['1959-01': '1959-03'].plot()

In [ ]:
descomposition.trend.plot()

## 4️⃣ Evaluación de estacionariedad

**4.1** Aplica la prueba de Dickey-Fuller aumentada a la serie original. ¿Es estacionaria?

In [ ]:
from statsmodels.tsa.stattools import adfuller

In [ ]:
adfuller(serie)[1] < 0.05

**4.2** Si no es estacionaria, aplica una primera diferenciación y repite la prueba.

In [ ]:
adfuller(serie.diff().dropna())[1] < 0.05

**4.3** Si sigue sin ser estacionaria, aplica una segunda diferenciación.

In [ ]:
adfuller(serie.diff().diff().dropna())[1] < 0.05

## 5️⃣ Identificación de parámetros (ACF y PACF)

**5.1** Grafica la ACF de la serie estacionaria. ¿Hasta qué rezago hay autocorrelación significativa?

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

In [ ]:
acf_plot = plot_acf(serie)

In [ ]:
from statsmodels.tsa.stattools import acf

In [ ]:
acf(serie)

**5.2** Grafica la PACF. ¿Qué rezagos parecen relevantes para el componente AR?

In [ ]:
from statsmodels.graphics.tsaplots import plot_pacf

In [ ]:
pacf_plot = plot_pacf(serie)

In [ ]:
from statsmodels.tsa.stattools import pacf

pacf(serie)

**5.3** Con base en ACF y PACF, sugiere posibles valores para **p** y **q**.

In [ ]:
p = 1
q = 1
d = 1

## 6️⃣ Ajuste de modelo ARIMA

**6.1** Ajusta un modelo ARIMA usando los valores estimados de p, d y q.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

In [ ]:
modelo_arima = ARIMA(serie, order = (1, 1, 1))
resultado_arima = modelo_arima.fit()

**6.2** Muestra el resumen del modelo ajustado.

In [ ]:
resultado_arima.summary()

**6.3** Grafica los residuos del modelo. ¿Se comportan como ruido blanco?

In [ ]:
resultado_arima.resid.plot()

## 7️⃣ Modelado estacional con SARIMA

**7.1** Evalúa visualmente si hay estacionalidad semanal (periodicidad = 7).

In [ ]:
descomposition = seasonal_decompose(df['Births'][:'1959-02'], model = 'additive', period = 7)
descomposition_plot = descomposition.plot()

**7.2** Si es necesario, aplica diferenciación estacional y evalúa estacionariedad.

In [ ]:
serie_dif_estacional = serie.diff(7)

In [ ]:
adfuller(serie_dif_estacional.diff().dropna())[1]<0.05

**7.3** Usa ACF y PACF en la serie diferenciada para estimar los valores de **P**, **D**, **Q**, y **s**.

In [ ]:
acf_plot = plot_acf(serie_dif_estacional.dropna())

In [ ]:
pacf_plot = plot_pacf(serie_dif_estacional.dropna())

**7.4** Ajusta un modelo SARIMA con los valores estimados.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
sarima_model = SARIMAX(serie, order = (0,1,0), seasonal_order = (1,1,1,7))
sarima_resultado = sarima_model.fit()

**7.5** Evalúa el ajuste y los residuos.

In [ ]:
residuos_sarima = sarima_resultado.resid
residuos_sarima.plot(title="Residuos del modelo SARIMA", figsize=(10, 4))

In [ ]:
residuos_sarima.mean()

## ✅ Conclusión


Reflexiona sobre lo siguiente:

- ¿Qué descubriste sobre la estructura de esta serie?
- ¿Qué modelo te pareció más adecuado: ARIMA o SARIMA?
- ¿Qué dificultades encontraste al identificar los parámetros?


In [ ]:
# Haz tu código acá